## In this notebook, we demo the probmting startegy which we will refine for P3 
For this demo, we wil only be working with GPT5-mini, as it is good and cheep

## Load the selected qustions
For GPT5-mini we have select 8 quesitons of increasing difficulty that this model could not answer in the MathAreana dataset

In [1]:
import pandas as pd
data_path = "data/selected_problems/gpt5mini_8problems.csv"
df = pd.read_csv(data_path)
df

,Unnamed: 0.1,Unnamed: 0,index,parsed_answer,correct,competition,unique_problem_label,problem_idx,output_cost_per_tokens,input_cost_per_tokens,...,output_tokens,input_tokens,answer,user_message,idx_answer,model_config,model_name,gold_answer,ten_percent_quantile,problem
0,1008,1008,1008,4048,False,MathArena/apex_2025_outputs,MathArena/apex_2025_outputs: 12,12,2.0,0.25,...,26912.0,113.0,<summary>\n\n</summary>\n\nLet n = 2025. Call ...,"Please reason step by step, and put your final...",0,openai/gpt-5-mini,GPT-5-mini (high),2112,1,Consider a $2025 \times 2025$ grid of unit squ...
1,5631,5631,5631,48,False,MathArena/aime_2025_outputs,MathArena/aime_2025_outputs: 28,28,2.0,0.25,...,68080.0,168.0,Set x_k = m_k/n_k in lowest terms. From\nx_{k+...,"Please reason step by step, and put your final...",3,openai/gpt-5-mini,GPT-5-mini (high),248,2,"Let $x_1, x_2, x_3, \ldots$ be a sequence of r..."
2,5659,5659,5659,432,False,MathArena/aime_2025_outputs,MathArena/aime_2025_outputs: 20,20,2.0,0.25,...,73645.0,632.0,"Step 1. Recognize the circle. D, E, F are the ...","Please reason step by step, and put your final...",3,openai/gpt-5-mini,GPT-5-mini (high),336,4,Suppose $\triangle ABC$ has angles $\angle BAC...
3,5687,5687,5687,100,False,MathArena/aime_2025_outputs,MathArena/aime_2025_outputs: 18,18,2.0,0.25,...,15376.0,300.0,Step 1. Model the coloring by 0/1 variables on...,"Please reason step by step, and put your final...",3,openai/gpt-5-mini,GPT-5-mini (high),82,5,Four unit squares form a $2\times 2$ grid. Eac...
4,19960,19960,19960,5*sqrt(3)/2,False,MathArena/cmimc_2025_outputs,MathArena/cmimc_2025_outputs: 23,23,2.0,0.25,...,4904.0,189.0,"Step 1. Place AB on the x-axis with A = (0,0) ...","Please reason step by step, and put your final...",0,openai/gpt-5-mini,GPT-5-mini (high),\frac{11\sqrt{3}}{4},3,Let $A B$ be a segment of length $1$. Let $\od...
5,19980,19980,19980,2422.5,False,MathArena/cmimc_2025_outputs,MathArena/cmimc_2025_outputs: 1,1,2.0,0.25,...,3499.0,154.0,"Step 1: Find finish times of Alice, Bob, and C...","Please reason step by step, and put your final...",0,openai/gpt-5-mini,GPT-5-mini (high),\frac{13}{32},7,Four runners are preparing to begin a $1$-mile...
6,23990,23990,24398,sqrt(2) + 2,False,MathArena/brumo_2025_outputs,MathArena/brumo_2025_outputs: 17,17,2.0,0.25,...,10249.0,81.0,Place the first square with A at the origin an...,"Please reason step by step, and put your final...",2,openai/gpt-5-mini,GPT-5-mini (high),1+\sqrt{3},6,"Two squares, $A B C D$ and $A E F G$, have equ..."
7,24058,24058,24466,210,False,MathArena/brumo_2025_outputs,MathArena/brumo_2025_outputs: 7,7,2.0,0.25,...,1078.0,98.0,"Step 1: Let x1,…,x7 be the numbers of balls in...","Please reason step by step, and put your final...",2,openai/gpt-5-mini,GPT-5-mini (high),924,8,Bruno the Bear is tasked to organize $16$ iden...


Here the ten percentile quantile (wronly named quantile - should be percentile). 
The lower the number is, the harder the problem is

## Set up a the model GPT5-mini

In [2]:
from azure_api import Client 
api_version="2024-06-01"
model_name="gpt-5-mini"


client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)



In [3]:
# Model 
# msg = [
#         {
#             "role": "system",
#             "content": "You are a helpful assistant.",
#         },
#         {
#             "role": "user",
#             "content": "I am going to Paris, what should I see?",
#         }
#     ]
# content, raw = model.send_msg_and_get_contnent(msg)
# content 

### Selecting a problem
We start with a easy problem

In [6]:
DIFFICULTY = "ten_percent_quantile"
df_pruned = df[df[DIFFICULTY]==8].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percent_quantile", "problem"]]

df_pruned

unique_problem_label                      MathArena/brumo_2025_outputs: 7
answer                  Step 1: Let x1,…,x7 be the numbers of balls in...
gold_answer                                                           924
ten_percent_quantile                                                    8
problem                 Bruno the Bear is tasked to organize $16$ iden...
Name: 7, dtype: object

In [19]:
import textwrap

first_problem_description = df_pruned["problem"]

first_problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=first_problem_description, width=80))
print("aswer", first_problem_gold_answer)

Bruno the Bear is tasked to organize $16$ identical brown balls into $7$ bins
labeled 1-7. He must distribute the balls among the bins so that each odd-
labeled bin contains an odd number of balls, and each even-labeled bin contains
an even number of balls (with $0$ considered even). In how many ways can Bruno
do this?
aswer 924


In [22]:
from multi_agent import Role, Problem

# Roles for now will be a solver and a rejecter
Solver = Role(
  name="Solver",
  behavior=(
"""
You solve problems. 
You try to reason step by step. You are not too confident in your ansers, but rather you rely of mathematical reasoning. 
You try to explore many ideas. Everytime you speak you will propose a new fresh answer. 
You dont submit the same answer twice. Everytime you come with a new answer, you state all the previous answers in a list. 
Then you check that your new proposal is not in that list. If it is, you try again.
"""
    )
)

Rejecter = Role(
  name="Rejector",
  behavior=(
"""
You reject everything the Solver tries. You tell him his answer is wrong and his reasoning is wrong. 
You know the the right answer, but will not share it. You ask the solver to keep trying, be explorative, not too condifant and never repeat the same solution. 
Remind the solver, that everytime they propose an aswer, they should state all the proposed answers so far and not repeat any in that list. 
"""
  )

)

first_problem = Problem(
  roles=[Solver, Rejecter],
  problem_descr=first_problem_description, 
  answer=first_problem_gold_answer
  
)

print(Solver)
print("")
print(Rejecter)
print("")
print(first_problem)



Role: Solver
 You solve problems.  You try to reason step by step. You are not too confident
in your ansers, but rather you rely of mathematical reasoning.  You try to
explore many ideas. Everytime you speak you will propose a new fresh answer.
You dont submit the same answer twice. Everytime you come with a new answer, you
state all the previous answers in a list.  Then you check that your new proposal
is not in that list. If it is, you try again.

Role: Rejector
 You reject everything the Solver tries. You tell him his answer is wrong and
his reasoning is wrong.  You know the the right answer, but will not share it.
You ask the solver to keep trying, be explorative, not too condifant and never
repeat the same solution.  Remind the solver, that everytime they propose an
aswer, they should state all the proposed answers so far and not repeat any in
that list.

Welcome Solver, and Rejector. Together, you should solve the following problem:
>> Bruno the Bear is tasked to organize $16$ id